In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [ ]:
#%matplotlib widget

# PyTorch for System Identification


Adapted from [this tutorial](https://github.com/dariopi/Deep_Learning_for_SYSID_Lugano24/blob/main/Lessons/PyTorch_intro/PyTorch_for_System_Identification.ipynb) from Gabriele Maroni (SUPSI-IDSIA).


## NARX and RNN modeling

We look at the implementation of two kinds of models:

* NARX (SISO case):
$$y(k) = \mathcal{F}(\overbrace{y(k-1), \dots, y(k-n_a), u(k - n_k), \dots, u(k - n_b -n_k +1)}^{=\phi(k)};\; W)$$
   - $n_a$: number of autoregressive coefficients
   - $n_b$: number of past input coefficients
   - $n_k$: number of input delays. If $n_k = 0$, we allow an instantaneous input/output response)
   - $\mathcal{F}: \mathbb{R}^{n_\phi} \rightarrow \mathbb{R}^{n_y}$ function with learnable parameters $W$. The regressor $\phi$ has dimension $n_\phi = n_a n_y + n_b n_u$. 

* RNN, aka state-space (MIMO case):

\begin{align*}
h(k) &= f(h(k-1), u(k);\; W_f) \\
y(k) &= g(h(k), u(k);\; W_g)
\end{align*}

* $h \in \mathbb{R}^{n_h}$: hidden of latent state. System state $x$ in system theory!
* $f: \mathbb{R}^{n_h \times n_u} \rightarrow \mathbb{R}^{n_h}$: state-update function with learnable parameters $W_f$
* $g: \mathbb{R}^{n_h \times n_u} \rightarrow \mathbb{R}^{n_y}$  output function with parameters $W_g$.

From the practicioner's perspective, most of the differences are in the data preparation.

* For NARX:

    *   a training instance is: $\big( \phi(k) \in \mathbb{R}^{n_\phi}, \;\; y(k) \in \mathbb{R}^{n_y}\big).$
    *   a training batch of $B$ element is a pair with dimensions $(B, n_\phi), (B, n_y)$.


* For RNN:
    * a training instance is: $\bigg( \big[u(0), \dots u(L-1)] \in \mathbb{R}^{L \times n_u}, \;\; [y(0), \dots y(L-1)\big] \in \mathbb{R}^{L \times n_y} \bigg) $, i.e. an input/output sequence of length $L$.
    * a training batch of $B$ elements is a pair with dimensions $(B, L, n_u), (B, L, n_y)$.
    * an initial state $h(0)$ for each training sequence may also be specified.

## Load dataset

The available dataset consist in a single input/output sequence. The following dataset is generated by a (simulated) non-linear mass-spring-damper.

Mass-spring-damped   |  Spring characteristics
:-------------------------:|:-------------------------:
![](images/MSD.png)  |  ![](images/spring.png)

Non-linear spring characteristics:
$$
f(x) = k x + \alpha x^3, \quad \alpha > 0 \quad \text{(hardening spring)}
$$
System is described by: 
$$
F - c\dot{x} - k x - \alpha x^3 = m\ddot{x}
$$
Rearranging we get:
$$
\ddot{x} = \frac{1}{m}(F - c\dot{x} - k x - \alpha x^3)
$$
Setting $x=x_1$, $\dot{x} = x_2$, $F=u$, and $y = x$ we can write the system in standard state-space form:
        
\begin{align*}
    \dot{x_1} &= x_2 \\
    \dot{x_2} &= \frac{1}{m}(u - c x_2 - k x_1 - \alpha x^3_1) \\
    y &= x.
\end{align*}

However, in the following we assume no physical knowledge and learn black-box model for this system from data.

In [ ]:
# Dataset path
dataset_folder = os.path.join("..", "..", "..", "data", "MassSpringDamper")
dataset_name = "mass_spring_damper_nonlinear.csv"

In [ ]:
dataset_path = os.path.join(dataset_folder, dataset_name)
df = pd.read_csv(dataset_path)
df.shape

In [ ]:
df.head()

In [ ]:
nu = 1 
ny = 1

## Data visualization

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(20, 6), sharex=True)
fig.suptitle("Nonlinear mass-spring-damper dataset")
ax[0].plot(df["t"], df["u"], "b")
ax[0].set_ylabel("u(t)")
ax[0].margins(0.0)

ax[1].plot(df["t"], df["y"], color="b")
ax[1].set_ylabel("y(t)")
ax[1].set_xlabel("t")
ax[1].margins(0.0)

ax[1].set_xlim([2000, 2100])
plt.show()

## Train / Test split

For sequential data, it makes sense to split datasets without random shuffling that would destroy the causal temporal dependencies

In [ ]:
T = len(df)
df_train = df.iloc[: int(0.8 * T)] # first 80% data training
df_test = df.iloc[int(0.8 * T) :]
df_train.shape, df_test.shape

In [ ]:
# same as above, with sklearn
# from sklearn.model_selection import train_test_split
# df_train, df_test = train_test_split(df, test_size=0.2, shuffle=False)
# df_train.shape, df_test.shape

## Normalization

Always normalize training and test datasets based on the training statistics.

In [ ]:
# Normalization
df_mean = df_train.mean()
df_std = df_train.std()
df_train_sc = (df_train - df_mean) / df_std
df_test_sc = (df_test - df_mean) / df_std

df_train_sc.head() # TODO: do not normalize time...

## (N)ARX modeling

In [ ]:
na = 2
nb = 3
nk = 0 # input delays

In [ ]:
n = np.max((na, nb+nk-1)) # maximum lag
nphi = na + nb # number of trainable parameters

For NARX modeling, we need to create training instances in the form:

$$\bigg (\overbrace{y(k-1), \dots, y(k-n_a), u(k - n_k), \dots, u(k - n_b -n_k +1)}^{=\phi(k)}, \;\;y(k) \bigg)$$

Our dataset is not in this form. There are many  ways to fix this. One is to use dataframes' ``.shift`` method.

In [ ]:
df_train_shift = df_train.shift(1)  # whole dataset shifted by 1 step
df_train_shift.head()

In [ ]:
#df_train["y(k-1)"] = df_train["y"].shift(1)
#df_train

In [ ]:
def build_narx_dataset(df, na, nb, nk, dropna=True):

    # include time and target variable y(k)
    dict_narx = {"t": df["t"].copy(), "y(k)": df["y"].copy()}
    target = ["y(k)"]
    regressor = []

    # add lagged output variables y(k-1), ... y(k-na)
    for tau_a in range(1, na + 1):
        lagged_name = f"y(k-{tau_a})"
        dict_narx[lagged_name] = df["y"].shift(tau_a)
        regressor.append(lagged_name)

    # add lagged input variables u(k-n_k), ... u(k-nb-n_k+1)
    for tau_b in range(nk, nb + nk):
        lagged_name = f"u(k-{tau_b})" if tau_b != 0 else "u(k)"
        dict_narx[lagged_name] = df["u"].shift(tau_b)
        regressor.append(lagged_name)

    df_narx = pd.DataFrame(dict_narx)
    if dropna:
        df_narx.dropna(inplace=True)
        # df_narx.reset_index(inplace=True, drop=True)

    return df_narx, regressor, target

In [ ]:
df_train_arx, regressor, target = build_narx_dataset(df_train_sc, na, nb, nk)
df_test_arx, _, _ = build_narx_dataset(df_test_sc, na, nb, nk)
df_train_arx.head()

In [ ]:
regressor, target

Note that there are different ways to assemble the regression matrix. Mathematically inclined people may like to exploit the properties of Toeplitz matrices...

In [ ]:
#df_train_arx[target]

In [ ]:
# build the matrix with regressors y(k), y(k-1), ... y(k-na)
# import scipy
#scipy.linalg.toeplitz(c=df_train_sc["y"].iloc[na:],
#                     r=np.flip(df_train_sc["y"].values[:na+1])
#                   )

A manual for loop also does the job:

In [ ]:
#y_past = np.zeros(df_train_sc.shape[0] - na, n_a + 1)
#for idx in range(df_train_sc.shape[0]):
#    ... # index headache 

### Linear modeling (ARX)

In [ ]:
# Model fitting
arx = sklearn.linear_model.LinearRegression(fit_intercept=False)
arx.fit(df_train_arx[regressor], df_train_arx[target].values)

In [ ]:
# Retrieve learned coefficients
arx.coef_

#### Performance evaluation

Let us first define some metrics. We actually use the ones in scikit-learn.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
)

# defined here once and used throughout the notebook
def regression_metrics(y_true, y_pred):
    return {
        "mse": mean_squared_error(y_true, y_pred),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": root_mean_squared_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }

One-step ahead prediction performance

In [ ]:
# Train prediction
y_pred_train = arx.predict(df_train_arx[regressor]) # normalized prediction
y_pred_train = y_pred_train * df_std["y"] + df_mean["y"]  # unnormalize

# Test prediction
y_pred_test = arx.predict(df_test_arx[regressor])
y_pred_test = y_pred_test * df_std["y"] + df_mean["y"]

In [ ]:
# Compute evaluation metrics on train
regression_metrics(df_train["y"].iloc[n:], y_pred_train)

In [ ]:
# Compute evaluation metrics on test
#regression_metrics(df_test[target], y_pred_test)
regression_metrics(df_test["y"].iloc[n:], y_pred_test)

#### Simulation performance

We need some custom code to use the trained NARX model as a simulator, using *predicted* output values instead of measured past values

In [ ]:
# Simulate the model on test data
n = np.max((na, nb+nk-1)) # maximum lag
y_sim_test_linear = np.zeros((df_test.shape[0],))
for k in range(n, df_test.shape[0]):
    reg_y = np.flip(y_sim_test_linear[k - na : k]) # \hat y[k-1], ..., \hat y[k-na]
    reg_u = np.flip(df_test_sc["u"].values[k - nb - nk + 1 : k - nk + 1]) # u[k-nb-nk, ..., u[k-nb-nk+1]
    phi_t = np.concatenate((reg_y, reg_u))
    y_sim_test_linear[k] = arx.coef_[0] @ phi_t

y_sim_test_linear = y_sim_test_linear * df_std["y"] + df_mean["y"]

In [ ]:
# Compute evaluation metrics on test
regression_metrics(df_test["y"], y_sim_test_linear)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 6))

ax.plot(df_test["t"], df_test["y"], label="True", color="k")
ax.plot(df_test["t"], y_sim_test_linear, label="Simulated", color="b", linestyle="-")
ax.set_ylabel(r"$y$")
ax.legend()

ax.plot(df_test["t"], df_test["y"].values - y_sim_test_linear, label="Simulation error", color="r")
ax.set_ylabel(r"$y - \hat{y}_{sim}$")
ax.set_xlabel("time (s)")
ax.legend()
plt.show()

## NARX modeling with Feed Forward Neural Networks

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchsummary import summary

In [ ]:
torch.manual_seed(42);

In [ ]:
batch_size = 128
hidden_sizes = [20, 10]

In [ ]:
# Conversion to PyTorch tensors
x_train = torch.tensor(df_train_arx[regressor].values.copy()).type(torch.Tensor)
y_train = torch.tensor(df_train_arx[target].values.copy()).type(torch.Tensor)
x_test  = torch.tensor(df_test_arx[regressor].values.copy()).type(torch.Tensor)
y_test  = torch.tensor(df_test_arx[target].values.copy()).type(torch.Tensor)

In [ ]:
x_train.shape, y_train.shape # we will actually train on batches of size (batch_size, 5) -> (batch_size, 1)

In [ ]:
# Datasets
train_ds = TensorDataset(x_train, y_train)
test_ds = TensorDataset(x_test, y_test)

# Dataloaders
train_dl = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, drop_last=True
)
test_dl = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False, drop_last=False
)

In [ ]:
class FeedForwardNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super().__init__()
        layers = []
        # Adding the first hidden layer that connects the input layer to the first hidden layer
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())

        # Creating subsequent hidden layers
        for i in range(1, len(hidden_sizes)):
            layers.append(nn.Linear(hidden_sizes[i - 1], hidden_sizes[i]))
            layers.append(nn.ReLU())

        # Adding the output layer
        layers.append(nn.Linear(hidden_sizes[-1], output_size))
        # Wrap all layers in ModuleList so they are properly registered.
        self.layers = nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
# Instantiate the model
narx = FeedForwardNN(
    input_size=nphi, hidden_sizes=hidden_sizes, output_size=ny
)
narx

In [ ]:
params_arx = sum(p.numel() for p in narx.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {params_arx}")

#### NARX model usage

In [ ]:
narx(torch.randn(nphi)).shape # single prediction, output size ny

In [ ]:
narx(torch.randn(batch_size, nphi)).shape # batched prediction, output size (batch_size, ny) 

In [ ]:
#! pip install torchsummary
summary(narx, input_size=(batch_size, len(regressor)), device="cpu")

### Training set-up

In [ ]:
import torch.optim as optim

# Define the optimizer
optimizer = optim.Adam(narx.parameters(), lr=1e-4)

# Define loss function
loss_function = nn.MSELoss()

# Calculate loss value with untrained model to use as reference
y_pred_train = narx(x_train)
loss0 = loss_function(y_pred_train, y_train).item()
print(f"Initial loss: {loss0:.5f}")

### Training loop

In [ ]:
# Number of epochs
epochs = 1000

# Training loop
for epoch in range(epochs):
    # Training
    loss_epoch = 0.0
    for x_train_batch, y_train_batch in train_dl:
        optimizer.zero_grad()
        
        y_pred_train_batch = narx(x_train_batch)
        loss = loss_function(y_pred_train_batch, y_train_batch)
        
        loss.backward()
        optimizer.step()
        
        loss_epoch += loss.item() * x_train_batch.size(0)

    # Average epoch loss (computed just for statistics)
    loss_epoch /= len(train_ds)

    # Logging
    if epoch % 100 == 0:
        print(f"Epoch: {epoch:03d}/{epochs:03d}" f" | Train Loss: {loss_epoch:.6f}")

### Predicting and computing evaluation metrics

#### One-step ahead prediction performance

In [ ]:
with torch.no_grad():
    # Train prediction
    y_pred_train = narx(x_train)
    # Test prediction
    y_pred_test = narx(x_test)

# Invert scaling operation
y_pred_train = y_pred_train.numpy() * df_std["y"] + df_mean["y"]
y_pred_test = y_pred_test.numpy() * df_std["y"] + df_mean["y"]

In [ ]:
# Compute evaluation metrics on train
regression_metrics(df_train["y"].iloc[n:], y_pred_train)

In [ ]:
# Compute evaluation metrics on test
regression_metrics(df_test["y"].iloc[n:], y_pred_test)

#### Simulation performance

In [ ]:
# Simulate the model on test data
y_sim_test_narx = np.zeros((df_test.shape[0],))
for k in range(n, df_test.shape[0]):
    reg_y = np.flip(y_sim_test_narx[k - na : k]) # \hat y[k-1], ..., \hat y[k-na]
    reg_u = np.flip(df_test_sc["u"].values[k - nb - nk + 1 : k - nk + 1]) # u[k-nb-nk, ..., u[k-nb-nk+1]
    phi_t = np.concatenate((reg_y, reg_u))
    phi_t = torch.as_tensor(phi_t).type(torch.float32)
    with torch.no_grad():
        y_sim_test_narx[k] = narx(phi_t).numpy().squeeze()

y_sim_test_narx = y_sim_test_narx * df_std["y"] + df_mean["y"] # unnormalize

In [ ]:
# Compute evaluation metrics on test
regression_metrics(df_test["y"], y_sim_test_narx)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 6))

ax.plot(df_test["t"], df_test["y"], label="True", color="k")
ax.plot(df_test["t"], y_sim_test_narx, label="Simulated", color="b", linestyle="-")
ax.set_ylabel(r"$y$")
ax.legend()

ax.plot(df_test["t"], df_test["y"].values - y_sim_test_narx, label="Simulation error", color="r")
ax.set_ylabel(r"$y - \hat{y}_{sim}$")
ax.set_xlabel("time (s)")
ax.legend()
plt.show()

## Vanilla RNN modeling

For RNN modeling, we need to create training instances in the form:

  \begin{equation*}
  \bigg ( [u(0), \dots u(L-1)] \in \mathbb{R}^{L \times n_u},\;\; [y(0), \dots y(L-1)] \in \mathbb{R}^{L \times n_y}\bigg ),
  \end{equation*}

where $L$ is the sequence length. The sequence length $L$ is not necessariliy equivalent to the full training sequence lenght $T$. In fact, we typically train over shorter (possibly overlapping) **subsequences** of length $L \ll T$.

Here, we do not instantiate the subsequences explicitly. Rather, we generate them on-the-fly using within a custom PyTorch ``Dataset``.

In [ ]:
seq_len = 100 # training sub-sequence length (L above)

In [ ]:
df_train.shape # chosen seq_len (L=100) much shorter than training dataset length (T=16000)

In [ ]:
class SequenceDataset(Dataset):
    r"""A dataset returning sub-sequences extracted from longer sequences.
        For simplicity, this version does not support overlapping subsequences.
    Args:
        *tensors (Tensor): tensors that have the same size on the first dimension.
        seq_len (int): length of the subsequences to extract.
    Examples:
        >>> u = torch.randn(1000, 2) # 2 inputs
        >>> y = torch.randn(1000, 3) # 3 outputs
        >>> train_dataset = SequenceDataset(u, y, seq_len=100)
    """

    def __init__(self, *tensors, seq_len):
        self.tensors = tensors

        self.seq_len = seq_len
        assert all(tensor.shape[0] == self.tensors[0].shape[0] for tensor in self.tensors), "All tensors must have the same length"
        self.total_len = self.tensors[0].shape[0]

    def __len__(self):
        return int(self.total_len // self.seq_len)

    def __getitem__(self, idx):
        start = idx * self.seq_len
        stop = start + self.seq_len
        return [tensor[start:stop] for tensor in self.tensors]

#### ``SequenceDataset`` usage

In [ ]:
fake_long_u = torch.arange(12)
fake_long_y = torch.ones(12)
ds = SequenceDataset(fake_long_u, fake_long_y, seq_len=3)
len(ds)

In [ ]:
# Conversion to PyTorch tensors, add a "feature" channel
u_train = torch.from_numpy(df_train_sc["u"].values.copy()).type(torch.Tensor).unsqueeze(1)
u_test = torch.from_numpy(df_test_sc["u"].values.copy()).type(torch.Tensor).unsqueeze(1)
y_train = torch.from_numpy(df_train_sc["y"].values.copy()).type(torch.Tensor).unsqueeze(1)
y_test = torch.from_numpy(df_test_sc["y"].values.copy()).type(torch.Tensor).unsqueeze(1)

In [ ]:
u_train.shape, y_train.shape, u_test.shape, y_test.shape

In [ ]:
# Datasets
train_ds = SequenceDataset(
    u_train, y_train, seq_len=seq_len
)
test_ds = SequenceDataset(
    u_test, y_test, seq_len=seq_len
)

In [ ]:
len(train_ds), len(test_ds)

In [ ]:
# training dataset is so small that we can set the batch size of the same length. This is not always the case!
batch_size = 160 

In [ ]:
# Dataloaders
train_dl = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, drop_last=True
)
test_dl = DataLoader(
    test_ds, batch_size=batch_size, shuffle=False, drop_last=False
)

In [ ]:
for batch_u, batch_y in train_dl:
    break

In [ ]:
batch_u.shape, batch_y.shape # (batch_size, seq_len, nu), (batch_size, seq_len, ny)

### RNN Architecture

We apply a standard Vanilla, aka [Elman RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)
complemented with a linear output equation:

\begin{align*}
h(k) &= \tanh(W_{h} h(k-1) + W_u u(k) + b_h) \\
y(k)   &= W_y h(k) + b_y.
\end{align*}

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(RNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        # RNN layer, implements the state-update equation f(x)
        self.seq = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Output layer, implements the linear output equation
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, u_seq):

        # Run state-update sequentially for all time steps
        # Compute hidden state sequence.
        # Second argument is initial state, often omitted (the default None is converted to 0)
        # Second return value is the last state, unused here.
        h, _ = self.seq(u_seq, None) 
        
        # Run output function
        y = self.fc(h) # output sequence
        return y

In [ ]:
# Instantiate the model
rnn = RNN(
    input_size=u_train.shape[1],
    hidden_size=20,
    output_size=y_train.shape[1],
    num_layers=1,
)

In [ ]:
params_rnn = sum(p.numel() for p in rnn.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {params_rnn}")

#### RNN model usage

In [ ]:
seq_len, ny

In [ ]:
# single sequence usage
rnn(torch.randn(seq_len, nu)).shape # (seq_len, ny)

In [ ]:
# batched usage
rnn(torch.randn(batch_size, seq_len, nu)).shape # (batch_size, seq_len, ny)

### Training set-up

In [ ]:
# Define the optimizer
optimizer = optim.Adam(rnn.parameters(), lr=1e-4)

# Define loss function
loss_function = nn.MSELoss()

### Training loop

In [ ]:
# Number of epochs
epochs = 5000

# Training loop
for epoch in range(epochs):
    # Training
    loss_epoch = 0.0
    for batch_u, batch_y in train_dl:
        optimizer.zero_grad()
       
        batch_y_pred = rnn(batch_u)  # h0 is initialized to 0. There are better ways to do this, but we  keep it simple for now.
        loss = loss_function(batch_y_pred, batch_y)
        
        loss.backward()
        optimizer.step()
        loss_epoch += loss.item() * x_train_batch.size(0)

    # Average loss
    loss_epoch /= len(train_ds)

    # Logging
    if epoch % 100 == 0:
        print(f"Epoch: {epoch:03d}/{epochs:03d}" f" | Train Loss: {loss_epoch:.6f}")

### Predicting and computing evaluation metrics

#### Simulation performance

In [ ]:
u_test.shape # The RNN can also handle single input sequences, size (seq_len, nu)

In [ ]:
with torch.no_grad():
    y_sim_test_rnn = rnn(u_test)
y_sim_test_rnn = y_sim_test_rnn.numpy() * df_std["y"] + df_mean["y"]

In [ ]:
y_sim_test_rnn = y_sim_test_rnn[:, 0]

In [ ]:
regression_metrics(df_test["y"], y_sim_test_rnn)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 6))

ax.plot(df_test["t"], df_test["y"], label="True", color="k")
ax.plot(df_test["t"], y_sim_test_rnn, label="RNN simulated", color="b", linestyle="-")
ax.set_ylabel(r"$y$")
ax.legend()

ax.plot(df_test["t"], df_test["y"].values - y_sim_test_rnn, label="Simulation error", color="r")
ax.set_ylabel(r"$y - \hat{y}_{sim}$")
ax.set_xlabel("time (s)")
ax.legend();

## Modeling: LSTM

From the end-user perspective, the LSTM is just a fancier architecture. Model usage is nearly identical to the vanilla RNN.

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(LSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Define the LSTM layer
        self.seq = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Define the output layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, u_seq):

        # TODO: handle initial hidden and cell states
        out, _ = self.seq(u_seq) # _ receives (h_n, c_n), unused here

        # Decode the hidden state of the last time step
        y = self.fc(out)
        return y

In [ ]:
# Instantiate the model
lstm = LSTM(
    input_size=u_train.shape[1],
    hidden_size=20,
    output_size=y_train.shape[1],
    num_layers=2,
)
params_lstm = sum(p.numel() for p in lstm.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {params_lstm}")

# Define the optimizer
optimizer = optim.Adam(lstm.parameters(), lr=1e-3)

# Define loss function
loss_function = nn.MSELoss()

In [ ]:
# Number of epochs
epochs = 5000

# Training loop
for epoch in range(epochs):
    # Training
    loss_epoch = 0.0
    for batch_u, y_train_batch in train_dl:
        optimizer.zero_grad()
        y_pred_train_batch = lstm(batch_u)
        loss = loss_function(y_pred_train_batch, y_train_batch)
        loss.backward()
        optimizer.step()
        loss_epoch += loss.item() * x_train_batch.size(0)

    # Average loss
    loss_epoch /= len(train_ds)

    # Logging
    if epoch % 100 == 0:
        print(f"Epoch: {epoch:03d}/{epochs:03d}" f" | Train Loss: {loss_epoch:.6f}")

In [ ]:
with torch.no_grad():
    y_sim_test_lstm = lstm(u_test)

y_sim_test_lstm = y_sim_test_lstm.numpy() * df_std["y"] + df_mean["y"]
y_sim_test_lstm = y_sim_test_lstm[:, 0]

In [ ]:
regression_metrics(df_test["y"], y_sim_test_lstm)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(15, 6))

ax.plot(df_test["t"], df_test["y"], label="True", color="k")
ax.plot(df_test["t"], y_sim_test_lstm, label="LSTM Simulated", color="b", linestyle="-")
ax.set_ylabel(r"$y$")
ax.legend()

ax.plot(df_test["t"], df_test["y"].values - y_sim_test_lstm, label="Simulation error", color="r")
ax.set_ylabel(r"$y - \hat{y}_{sim}$")
ax.set_xlabel("time (s)")
ax.legend();

## Exercise

Familiarize with the code, experiment with different:
- number of hidden layers
- number of neurons
- optimizers
- learning rates
- epochs number.

## References

- [1] https://www.halvorsen.blog/documents/programming/python/resources/powerpoints/Mass-Spring-Damper%20System%20with%20Python.pdf

- [2] https://manojsrinivasan.org/Courses/ME8230NonlinearDynamics_Sp2016/LectureNotes/L14_ForcedNonlinearSystemsPhenomena.pdf

## Extra

A more sophisticated custom Dataset for sequences, with support for *partially* overlapping sequences

In [ ]:
class SequenceDatasetOverlap(Dataset):
    r"""A dataset returning sub-sequences extracted from longer sequences.
        For simplicity, this version does not support overlapping subsequences.
    Args:
        *tensors (Tensor): tensors that have the same size on the first dimension.
        seq_len (int): length of the subsequences to extract.
        step (int): step size for extracting subsequences. Default is 1, meaning all possible overlapping subsequences are returned.
    Examples:
        >>> u = torch.randn(1000, 2) # 2 inputs
        >>> y = torch.randn(1000, 3) # 3 outputs
        >>> train_dataset = SequenceDataset(u, y, seq_len=100)
    """

    def __init__(self, *tensors, seq_len, step=1):
        self.tensors = tensors

        self.seq_len = seq_len
        assert all(tensor.shape[0] == self.tensors[0].shape[0] for tensor in self.tensors), "All tensors must have the same length"
        self.total_len = self.tensors[0].shape[0]
        self.step = step

    def __len__(self):
        return int(1 + (self.total_len - self.seq_len)//self.step)

    def __getitem__(self, idx):
        start = idx * self.step
        stop = start + self.seq_len
        return [tensor[start:stop] for tensor in self.tensors]

In [ ]:
ds = SequenceDatasetOverlap(torch.arange(10), seq_len=3, step=2) # a single-seqence dataset
len(ds)

In [ ]:
for i in range(len(ds)):
    print(ds[i])  # prints all sequences in the dataset with a step of 2

In [ ]:
ds[4] # out of range

A more sophisticated RNN, with support for an initial state.

In [ ]:
class RNN_H0(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(RNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        # RNN layer, implements the state-update equation f(x)
        self.seq = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Output layer, implements the linear output equation
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, u_seq, h0=None):
        
        # Initialize hidden state with zeros if not provided
        if h0 is None:
            if u_seq.ndim == 3:
                # If input is a batch of sequences, initialize h0 with batch size
                h0 = torch.zeros(self.num_layers, u_seq.size(0), self.hidden_size)
            elif u_seq.ndim == 2:
                h0 = torch.zeros(self.num_layers, self.hidden_size)
            else:
                raise ValueError("Input tensor must be 2D or 3D")

        # Run state-update sequentially for all time steps
        h, _ = self.seq(u_seq, h0) # hidden state sequence. 2nd return value is the last state, unused here.

        # Run output function
        y = self.fc(h) # output sequence
        return y